In [11]:
# paramètres : je les ai réunis ici afin d'avoir un code 'modulaire' : on peut choisir ce qu'on veut
# observer en fonction de paramètres listés ici

# pour le choix du matériau
cle_materiau = "mp-570213"

cle_utilisateur = "rEVEIFsc61iExKZbJ2QyepniwSgJ1m6W"

# le nombre de décimales qu'on veut voir pour les valeurs chiffrées
dec = 4

In [12]:
# téléchargement

from pymatgen.symmetry import analyzer as az
from pymatgen.ext.matproj import MPRester
import numpy as np
from IPython.display import Image

with MPRester(cle_utilisateur) as m:
    structure = m.get_structure_by_material_id(cle_materiau)
    structure = az.SpacegroupAnalyzer(structure).get_primitive_standard_structure()

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
# petite fonction utilitaire avec exemple

def give_coord(A):
    """
    détermine les nouvelles coordonnées d'un point (x,y,z)
    en fonction de la matrice de rotation A:
    
    A_ij appartient a {1,0,-1}
    
                    1 0 0
    résultat si A = 0 1 0  : ['x','y','z']
                    0 0 1
    """
    x = ['x','y','z']
    y = ['']*3
    index = [0,1,2] 
    for i in index:
        for j in index:
            if A[i][j] == 1:
                if y[j] == '':
                    y[j] = x[i]
                else :
                    y[j] += '+'+x[i]
                
            if A[i][j] == -1:
                y[j] += '-'+ x[i]
    return y


# exemple

A = np.zeros((3,3))
A[0][1] = -1
A[1][0] = -1
A[2][2] = 1
print('exemple d\'utilisation :')
print('si A =')
print(A,end='\n\n')
print('le résultat est '+str(give_coord(A)))

exemple d'utilisation :
si A =
[[ 0. -1.  0.]
 [-1.  0.  0.]
 [ 0.  0.  1.]]

le résultat est ['-y', '-x', 'z']


In [14]:
an = az.SpacegroupAnalyzer(structure)
structure = an.get_conventional_standard_structure()
an = az.SpacegroupAnalyzer(structure)


# on récupère les informations de site sur 3 atomes différents
all_sites = structure.sites
sites = [all_sites[1], all_sites[3], all_sites[6]]

for site in sites:
    print('position de l\'atome "'+str(site.specie)+'" : '+str(np.around(site.frac_coords,dec)))
    

# on prend 3 opérations de symétries différentes de l'identité et qui ne sont pas l'opposé d'une opération déjà utilisée
all_symetries = an.get_symmetry_operations()
symetries = [all_symetries[7], all_symetries[12], all_symetries[25]]


# on fait un bel affichage
i = 0
for s in symetries :
    i+=1 
    rot_matrix = s.rotation_matrix
    tr_vector = s.translation_vector
    
    print('\n\nsymétrie '+str(i)+':')
    print('matrice de rotation :')
    print(np.around(rot_matrix),end='\n\n')
    print(give_coord(rot_matrix))
    print('\nvecteur de translation : '+str(np.around(tr_vector,dec))+'\n')
    
    for site in sites :
        new_coord = np.dot(site.frac_coords, rot_matrix)+tr_vector

position de l'atome "Li" : [0.  0.5 0.5]
position de l'atome "Li" : [0.5 0.5 0. ]
position de l'atome "Mg" : [0.  0.  0.5]


symétrie 1:
matrice de rotation :
[[ 0. -1.  0.]
 [-1.  0.  0.]
 [ 0.  0.  1.]]

['-y', '-x', 'z']

vecteur de translation : [0. 0. 0.]



symétrie 2:
matrice de rotation :
[[ 0.  0. -1.]
 [ 1.  0.  0.]
 [ 0. -1.  0.]]

['y', '-z', '-x']

vecteur de translation : [0. 0. 0.]



symétrie 3:
matrice de rotation :
[[ 0.  1.  0.]
 [-1.  0.  0.]
 [ 0.  0. -1.]]

['-y', 'x', '-z']

vecteur de translation : [0.5 0.5 0. ]

